In [ ]:
%run ./audit_utils

In [ ]:
update_audit_table("EV_Data", progress="Ingest_Gold", status="In_Progress", start=True)

In [ ]:
%python
# Define the database and table names
database_name = "silver"
table_name = "vehicle_data"

# Query the data from the managed table
df = spark.sql(f"SELECT * FROM {database_name}.{table_name}")
df.display()

In [ ]:
#Which one of the car make is more efficient?
gold_df = spark.sql(
    f"""
    SELECT Model_Year, County, City, State, Electric_Vehicle_type, Make, Model,
           COUNT(DOL_Vehicle_ID) ev_cnt, AVG(Electric_Range) AS Avg_ev_Range
    FROM {database_name}.{table_name}
    GROUP BY Model_Year, County, City, State, Electric_Vehicle_type, Make, Model
    """)

display(gold_df)

In [ ]:
%sql
create schema if not exists de_demo.gold;

In [ ]:
gold_db = "gold"
gold_table = "ev_insights"

gold_df.write.mode("overwrite").saveAsTable(f"{gold_db}.{gold_table}")

display(gold_df)

In [ ]:
%sql
select * from de_demo.gold.ev_insights

In [ ]:
%python
# Is there any relationship between the choice of EV make and city?
# Group by 'City' and 'Make' to see the count of each make in different cities
df_city_make = df.groupBy("City", "Make").count().orderBy("count", ascending=False)

# Show the relationship between city and make
df_city_make.show()

In [ ]:
%python
# Which Plug-in Hybrid Electric Vehicle (PHEV) is preferred by buyers?
# Filter for Plug-in Hybrid Electric Vehicle (PHEV)
df_phev = df.filter(col("Electric_Vehicle_Type").contains("Plug-in Hybrid Electric Vehicle (PHEV)"))

# Group by 'Make' and 'Model' to see the count of each PHEV
df_phev_preference = df_phev.groupBy("Make", "Model").count().orderBy("count", ascending=False)

# Show the most preferred PHEV by buyers
df_phev_preference.show()

In [ ]:
%python
# Based on the data, which car make and model would you recommend?
from pyspark.sql.functions import avg, count
df = df.withColumn("Base_MSRP", col("Base_MSRP").cast("int"))
df_recommend = df.groupBy("Make", "Model").agg(
    avg("Electric_Range").alias("Avg_Electric_Range"),
    avg("Base_MSRP").alias("Avg_Base_MSRP"),
    count("DOL_Vehicle_ID").alias("Popularity")
).orderBy("Avg_Electric_Range", ascending=False)

df_recommend.show()

In [ ]:

update_audit_table("EV_Data", progress="Ingest_Gold", status="Completed", end=True)